# Consensus clustering

Monti-style consensus KMeans clustering of the appendiceal neoplasm methylation cohort.

1. Load the beta matrix and remove the held-aside samples (NEN + duplicates).
2. Select the top-N most variable CpGs (SD of M-values), then drop redundant CpGs
   (of any two with |r| > `max_cpg_correlation`, the more variable one is kept).
3. Consensus clustering: `n_bootstraps` subsamples of `subsample_ratio` of the samples, KMeans for
   every k in `k_range`, consensus C = (times co-clustered) / (times co-sampled).
4. Choose k by the consensus CDF and PAC (proportion of ambiguous clustering, consensus in (0.1, 0.9)).
5. Final labels: average-linkage hierarchical clustering on 1 - C at the PAC-minimising k.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering

## Config

In [ ]:
CONFIG = {
    "beta_csv": "./results/betas_corrected.csv",  # CpGs x samples
    "out_dir": "./results/clustering",

    # feature selection
    "n_top_features": 5000,         # top-SD CpGs, ranked on M-values
    "max_cpg_correlation": 0.9,     # 1.0 = no correlation filter

    # samples excluded from clustering: NEN + duplicates
    "hold_aside_samples": [
        "207899840050_R01C01", "207899840050_R02C01", "207899840050_R03C01", "207899840050_R04C01", "207899840050_R05C01", "207899840050_R06C01", "207899840050_R07C01", "207899840005_R01C01", "207899840055_R01C01", "207899840055_R02C01",
        "207899840055_R03C01","207899840055_R04C01","207899840055_R05C01","207899840055_R06C01","207899840062_R01C01","207899840062_R02C01","207899840062_R03C01","207899840062_R04C01","207899840062_R05C01", "207899840055_R07C01", "208061540032_R01C01",
        "207961480030_R02C01", "207899840043_R07C01", "207899840043_R08C01", "207899840054_R01C01", "207899840054_R02C01", "207899840054_R03C01", "207899840054_R04C01", "207899840054_R05C01", "207899840054_R06C01"
    ],

    # consensus clustering
    "k_range": list(range(2, 8)),
    "n_bootstraps": 250,
    "subsample_ratio": 0.8,
    "random_state": 42,
}
os.makedirs(CONFIG["out_dir"], exist_ok=True)

## Data and CpG selection

CpGs are ranked on M-values (M = log2(beta / (1 - beta))), clustering runs on the beta values of the
selected CpGs.

In [ ]:
betas = pd.read_csv(CONFIG["beta_csv"], index_col=0, low_memory=False).T   # samples x CpGs
betas = betas.dropna(axis=1, how="any")
held = [s for s in CONFIG["hold_aside_samples"] if s in betas.index]
betas = betas.drop(index=held)

clipped = betas.clip(1e-6, 1 - 1e-6)
mvals = np.log2(clipped / (1 - clipped))

# top-N CpGs by SD of the M-values, most variable first
sd = mvals.std(axis=0, ddof=0)
top = sd.nlargest(CONFIG["n_top_features"]).index

# correlation filter: walk from most to least variable, drop a CpG if |r| with a kept CpG is too high
Z = mvals[top].values
Z = Z - Z.mean(axis=0)
Z = Z / np.linalg.norm(Z, axis=0)                  # now r(i, j) = Z[:, i] @ Z[:, j]
kept = []
for j in range(Z.shape[1]):
    if kept and np.abs(Z[:, kept].T @ Z[:, j]).max() > CONFIG["max_cpg_correlation"]:
        continue
    kept.append(j)
cpgs = top[kept]

X = betas[cpgs]
print(f"{X.shape[0]} samples clustered ({len(held)} held aside), "
      f"{len(cpgs)} of the top {len(top)} CpGs kept after the correlation filter")

## Consensus clustering

All k share the same bootstrap draws. KMeans uses `n_init=1`; the resampling supplies the variation.

In [ ]:
ks = CONFIG["k_range"]
seed = CONFIG["random_state"]
n = X.shape[0]
size = int(n * CONFIG["subsample_ratio"])
rng = np.random.default_rng(seed)

M = {k: np.zeros((n, n)) for k in ks}   # times each pair was clustered together
I = np.zeros((n, n))                    # times each pair was sampled together
for b in range(CONFIG["n_bootstraps"]):
    idx = rng.choice(n, size=size, replace=False)
    I[np.ix_(idx, idx)] += 1
    for k in ks:
        lab = KMeans(n_clusters=k, n_init=1, random_state=seed + b).fit_predict(X.values[idx])
        M[k][np.ix_(idx, idx)] += lab[:, None] == lab[None, :]

consensus = {}
for k in ks:
    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.nan_to_num(M[k] / I)
    np.fill_diagonal(C, 1.0)
    consensus[k] = pd.DataFrame(C, index=X.index, columns=X.index)

## Consensus CDF and PAC

PAC = fraction of sample pairs with consensus in (0.1, 0.9). A perfectly stable partition has a
step-shaped CDF and PAC = 0, so the k with the lowest PAC is chosen.

In [ ]:
iu = np.triu_indices(n, k=1)
cdf, pac = {}, {}
for k in ks:
    v = np.sort(consensus[k].values[iu])
    cdf[k] = v
    pac[k] = np.mean((v > 0.1) & (v < 0.9))

pac = pd.Series(pac, name="PAC").rename_axis("k")
best_k = int(pac.idxmin())
print(pac.round(4).to_string())
print(f"best k (min PAC) = {best_k}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
y = np.arange(1, len(iu[0]) + 1) / len(iu[0])
for k in ks:
    ax[0].plot(cdf[k], y, label=f"k={k}")
ax[0].axvspan(0.1, 0.9, color="grey", alpha=0.15)
ax[0].set(xlabel="consensus index", ylabel="CDF", title="Consensus CDF")
ax[0].legend()
ax[1].plot(pac.index, pac.values, marker="o", color="tab:red")
ax[1].axvline(best_k, color="black", linestyle="--", alpha=0.6)
ax[1].set(xlabel="k", ylabel="PAC", xticks=ks, title=f"PAC (lower = better), best k = {best_k}")
plt.tight_layout()
for ext in ["png", "pdf"]:
    plt.savefig(os.path.join(CONFIG["out_dir"], f"consensus_cdf_pac.{ext}"), dpi=150, bbox_inches="tight")
plt.show()

pac.to_csv(os.path.join(CONFIG["out_dir"], "pac.csv"))
pd.DataFrame({f"k{k}": cdf[k] for k in ks}).assign(cdf=y).to_csv(
    os.path.join(CONFIG["out_dir"], "consensus_cdf.csv"), index=False)

## Cluster labels at the best k

In [ ]:
labels = AgglomerativeClustering(n_clusters=best_k, metric="precomputed", linkage="average") \
    .fit_predict(1 - consensus[best_k].values)
labels = pd.Series(labels, index=X.index, name="consensus_label")
print(labels.value_counts().sort_index().to_dict())

labels.to_csv(os.path.join(CONFIG["out_dir"], "consensus_labels.csv"))
pd.Series(cpgs, name="cpg").to_csv(os.path.join(CONFIG["out_dir"], "consensus_cpgs.csv"), index=False)